<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_29_practicum_data_structures/note_lesson_29_structures_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧱 Урок 29 — Практикум П6: структури даних диспетчерської → пакет `dispatch`

П'ять питань диспетчерської «Смачно + Таксі» — п'ять структур. Кожну спершу проєктуємо (**картка класу**: що за річ, що знає, що вміє, які правила, чого не робить), потім пишемо повністю й дивимось, як змінюється її стан крок за кроком.

| Розділ | Питання диспетчерської | Структура | Приклади | Задача |
|---|---|---|---|---|
| 0 | одна помилка для всіх структур | `EmptyError` | 0.1 | |
| 1 | скасувати останнє призначення | стек `Stack` | 1.1–1.4 | **Задача 1** — перевірка дужок у шаблоні SMS |
| 2 | хто прийшов першим | черга `Queue`, `deque` | 2.1–2.5 | |
| 3 | найтерміновіше замовлення | купа `MinHeap`, `heapq` | 3.1–3.8 | |
| 4 | автодоповнення адрес | префіксне дерево `Trie` | 4.1–4.6 | |
| 5 | ціни маршрутів без повторних запитів | `OrderedDict`, `lru_cache`, LRU-кеш | 5.1–5.4 | **Задача 2** — клас `LRUCache` |
| 6 | усе разом | пакет `dispatch` і його 57 тестів | 6.1–6.4 | **Задача 3** — рівні дедлайни |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`class`, `def`, константи) лише створює їх — **запуск завжди в наступній клітинці**. Так приклад можна перезапускати, не перевизначаючи класи.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час у вимірах у тебе буде іншим — важливе **співвідношення**: «учетверо довше», а не «0,8 с».
- Теорія й покрокові схеми — у книзі: [Урок 29](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_29/).

---
## 0. Одна помилка для всіх структур

Кожна структура сьогодні вміє «взяти» елемент: `pop`, `dequeue`, `peek`. Що робити, коли брати нічого? Усі кидатимуть **одну** помилку — `EmptyError`. Вона наслідується від `LookupError` — батька `IndexError` і `KeyError` (урок 14): хто вже ловить «не знайшов», зловить і нашу помилку.

In [ ]:
import heapq
import itertools
import random
import time
from collections import OrderedDict, deque
from functools import lru_cache


class EmptyError(LookupError):
    """Спроба взяти елемент з порожньої структури."""

### Приклад 0.1. Хто зловить `EmptyError`

Задача: переконатися, що код, який уже ловить «не знайшов» через `except LookupError`, без змін зловить і нашу помилку. Кидаємо три різні винятки й ловимо їх **одним** `except`.

In [ ]:
def take_from_empty(kind):
    """Спробувати взяти елемент з порожньої структури потрібного виду."""
    if kind == "список":
        return [].pop()
    if kind == "словник":
        return {}["Поділ"]
    raise EmptyError("стек порожній")

In [ ]:
for base in (EmptyError, IndexError, KeyError):
    print(f"{base.__name__:<10} — нащадок LookupError: {issubclass(base, LookupError)}")

for kind in ("список", "словник", "наш стек"):
    try:
        take_from_empty(kind)
    except LookupError as error:
        print(f"{kind:<8}: зловили {type(error).__name__:<10} → {error}")

Один `except LookupError` упорався з усіма трьома: порожній список (`IndexError`), відсутній ключ (`KeyError`) і наша `EmptyError`. **Правило:** власний виняток наслідуй від найближчого за змістом стандартного — тоді він природно вбудовується в існуючий код.

---
## 1. Стек: скасувати останнє

Диспетчер призначає замовлення кур'єрам і іноді помиляється. Кнопка «Скасувати» має відкотити **останню** дію, потім передостанню — як Ctrl+Z у редакторі. Це **стек** (LIFO, last in — first out): стос тарілок, де беруть верхню.

```text
push Оксана·104      push Тарас·103       push Ігор·101        pop → Ігор·101
[Оксана·104]         [Оксана·104,         [Оксана·104,         [Оксана·104,
                      Тарас·103]           Тарас·103,           Тарас·103]
                                           Ігор·101]  ← вершина
```

Вершина — **кінець** списку: там `append` і `pop()` коштують `O(1)`.

### Приклад 1.1. Стек на голому списку

Задача: побачити LIFO без жодних класів. Список уже вміє все потрібне: `append` кладе на вершину, `pop()` знімає з вершини.

In [ ]:
actions = []
for action in [("Оксана", 104), ("Тарас", 103), ("Ігор", 101)]:
    actions.append(action)
    print("push", action, "→", actions)

while actions:
    print("pop ", actions.pop(), "→", actions)

Зняли у зворотному порядку: Ігор, Тарас, Оксана. Але голий список дозволяє й зайве — `actions.pop(0)`, `actions.insert(1, …)`, `actions.sort()`. Будь-що з цього ламає сенс «скасувати останнє». Тому загорнемо список у клас, який дає **лише** операції стеку.

**Картка класу `Stack`**

| Запитання | Відповідь | У коді |
|---|---|---|
| Що це за річ? | стос дій, де доступна лише верхня | `class Stack` |
| Що знає? | елементи в порядку додавання | прихований список `self._items` |
| Що вміє? | покласти, зняти, подивитися верхній; довжина, «чи порожній», обхід | `push`, `pop`, `peek`, `__len__`, `__bool__`, `__iter__` |
| Які правила тримає? | брати можна лише вершину; з порожнього — `EmptyError` | перевірка в `pop` і `peek` |
| Чого не робить? | не дає доступу до середини, не сортує, не знає про замовлення | немає `__getitem__`, `sort` |

### Приклад 1.2. Клас `Stack`

Визначення — у цій клітинці, запуск — у наступній. `__iter__` віддає елементи **від вершини до дна** — у тому порядку, в якому їх діставатиме `pop()`.

In [ ]:
class Stack:
    """Стек (LIFO): останнім поклали — першим узяли. Усі операції O(1)."""

    def __init__(self, items=()):
        self._items = list(items)       # вершина стеку — кінець списку

    def push(self, item):
        self._items.append(item)

    def pop(self):
        if not self._items:
            raise EmptyError("стек порожній")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise EmptyError("стек порожній")
        return self._items[-1]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __iter__(self):
        """Від вершини до дна — у тому порядку, в якому їх діставатиме pop()."""
        return reversed(self._items)

    def __repr__(self):
        return f"Stack({self._items!r})"

In [ ]:
history = Stack()
for action in [("Оксана", 104), ("Тарас", 103), ("Ігор", 101)]:
    history.push(action)
    print("push", action, "→", history)

print("від вершини до дна:", list(history))
print("скасовуємо:", history.pop())
print("наступним скасуємо:", history.peek(), "| у стеку:", len(history))

assert history.peek() == ("Тарас", 103) and len(history) == 2
assert list(Stack([1, 2, 3])) == [3, 2, 1]

`repr` показує внутрішній список (вершина праворуч), а `list(history)` — порядок, у якому дії скасовуватимуться (вершина першою). `peek` лише дивиться: після нього в стеку так само 2 дії.

Зверни увагу на **інкапсуляцію** (урок 22): список прихований у `_items`, а назовні — лише операції стеку.

### Приклад 1.3. Порожній стек і `if stack:`

Задача: що буде, якщо скасовувати частіше, ніж призначали? Завдяки `__bool__` стек можна перевіряти як список — `while history:`. А `pop` з порожнього кидає `EmptyError`, а не загадкову `IndexError: pop from empty list`.

**🔮 Прогноз:** у стеку дві дії, а цикл нижче тричі натискає «Скасувати». Що надрукує третє натискання?

<details>
<summary>Відповідь</summary>

`нема чого скасовувати: стек порожній`. Перші два `pop` знімуть Тараса й Оксану, третій кине `EmptyError`, і `except` її перехопить.

</details>

In [ ]:
history = Stack([("Оксана", 104), ("Тарас", 103)])
for click in range(1, 4):
    try:
        print(f"натискання {click}: скасовано {history.pop()}")
    except EmptyError as error:
        print(f"натискання {click}: нема чого скасовувати: {error}")

print("стек порожній?", not history, "| bool(Stack()):", bool(Stack()))
for take in (Stack.pop, Stack.peek):
    try:
        take(Stack())
        raise AssertionError("порожній стек мав кинути EmptyError")
    except EmptyError:
        pass

Третє натискання не «впало», а отримало зрозуміле повідомлення. `not history` — `True`: Python викликав наш `__bool__`.

### Приклад 1.4. Скасувати й повторити: два стеки

Задача: крім «Скасувати» (Ctrl+Z) потрібна «Повторити» (Ctrl+Y). Невелика зміна — **другий** стек: скасована дія не зникає, а перекладається в `redo`. Нова дія очищає `redo` — повторювати старе після неї вже не можна (так працює будь-який редактор).

```text
призначити → done.push         скасувати → done.pop → redo.push
нова дія   → redo очищається   повторити → redo.pop → done.push
```

In [ ]:
class AssignmentBoard:
    """Дошка призначень зі «Скасувати» і «Повторити» на двох стеках."""

    def __init__(self):
        self._done = Stack()
        self._redo = Stack()

    def assign(self, courier, order_id):
        self._done.push((courier, order_id))
        self._redo = Stack()                 # нова дія — історія «повторити» зникає

    def undo(self):
        action = self._done.pop()
        self._redo.push(action)
        return action

    def redo(self):
        action = self._redo.pop()
        self._done.push(action)
        return action

    def __repr__(self):
        return f"виконано={list(reversed(list(self._done)))} | можна повторити={list(self._redo)}"

In [ ]:
board = AssignmentBoard()
board.assign("Оксана", 104)
board.assign("Тарас", 103)
print("2 призначення:  ", board)
print("скасувати      →", board.undo(), "|", board)
print("повторити      →", board.redo(), "|", board)
print("скасувати      →", board.undo(), "|", board)
board.assign("Ігор", 101)
print("нова дія Ігор:  ", board)
try:
    board.redo()
except EmptyError:
    print("повторити      → нема чого: нова дія очистила redo")

Після скасування Тарас переїхав у `redo` і повернувся звідти «Повторити». Коли ж після скасування з'явилась нова дія (Ігор), повторювати Тараса вже не можна. **Правило:** дві протилежні операції «скасувати/повторити» — це два стеки, що перекладають елемент один одному.

### Задача 1. Перевірка шаблону SMS

Шаблони повідомлень клієнтам містять дужки: `"Замовлення {id} (кур'єр: {name}) [до {time}]"`. Напиши `is_balanced(text)`: кожна дужка `(`, `[`, `{` закрита відповідною і в правильному порядку. Використай `Stack`:

- відкривну дужку — у стек;
- закривна має відповідати **верхній** відкривній: `pop()` і порівняй через `PAIRS`; якщо стек порожній — зайва закривна;
- наприкінці стек має бути порожнім (інакше щось не закрито).

Перевірка — у наступній клітинці.

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_balanced(text):
    # YOUR CODE HERE
    pass


In [ ]:
for template in ["Замовлення {id} (кур'єр: {name}) [до {time}]", "({)}", "((", "())", "без дужок"]:
    print(f"{template!r:50} → {is_balanced(template)}")

assert is_balanced("Замовлення {id} (кур'єр: {name}) [до {time}]")
assert not is_balanced("({)}"), "порядок: ) має закривати верхню відкривну, а там {"
assert not is_balanced("(("), "наприкінці стек не порожній — дужки не закриті"
assert not is_balanced("())"), "зайва закривна: стек порожній, а закривна прийшла"
assert is_balanced("без дужок")
print("✅ Задача 1 розв'язана")

---
## 2. Черга: хто перший прийшов

Звичайні замовлення обслуговують **по черзі**: хто раніше замовив, того раніше й везуть. Це **черга** (FIFO, first in — first out).

```text
enqueue 101, 102, 105 — у хвіст          dequeue — з голови
голова → [101, 102, 105] ← хвіст         101 → кур'єру;  [102, 105]
```

Чому не взяти `list` і не знімати з початку через `pop(0)`? Поміряймо.

### Приклад 2.1. Дослід подвоєння: `list.pop(0)` проти `deque.popleft()`

**🔮 Прогноз:** `list.pop(0)` для 50 000 елементів займає близько 0,2 с. Скільки займе для 100 000? А `deque.popleft()`?

<details>
<summary>Відповідь</summary>

`list.pop(0)` — приблизно **учетверо** довше: кожен виклик зсуває всі елементи (`O(n)`), а викликів теж удвічі більше. `deque.popleft()` — `O(1)`, тож лише **удвічі** довше, і в сотні разів швидше за список.

</details>

In [ ]:
def drain_list(n):
    """Скільки секунд іде на те, щоб зняти з початку списку всі n елементів."""
    items = list(range(n))
    start = time.perf_counter()
    while items:
        items.pop(0)
    return time.perf_counter() - start


def drain_deque(n):
    queue = deque(range(n))
    start = time.perf_counter()
    while queue:
        queue.popleft()
    return time.perf_counter() - start

In [ ]:
previous = None
for n in (50_000, 100_000):
    slow, fast = drain_list(n), drain_deque(n)
    growth = f" (×{slow / previous:.1f} до попереднього)" if previous else ""
    print(f"{n:>7}: list.pop(0) {slow:.2f} с{growth}, deque.popleft() {fast:.3f} с — у {slow / fast:.0f} разів швидше")
    previous = slow

Удвічі більше даних — у `list` приблизно вчетверо довше: кожен `pop(0)` зсуває всі наступні елементи (`O(n)` на операцію, `O(n²)` на всю чергу). У `collections.deque` («двобічна черга») обидва кінці — `O(1)`, тож час росте лише вдвічі.

**Картка класу `Queue`**

| Запитання | Відповідь | У коді |
|---|---|---|
| Що це за річ? | черга замовлень: хто раніше прийшов, того раніше обслужать | `class Queue` |
| Що знає? | елементи від голови до хвоста | прихована `deque` у `self._items` |
| Що вміє? | стати в хвіст, вийти з голови, подивитися голову; довжина, «чи порожня», обхід | `enqueue`, `dequeue`, `peek`, `__len__`, `__bool__`, `__iter__` |
| Які правила тримає? | ставати — лише в хвіст, виходити — лише з голови; з порожньої — `EmptyError` | перевірка в `dequeue`, `peek` |
| Чого не робить? | не пропускає без черги, не сортує за терміновістю | для терміновості — купа (розділ 3) |

### Приклад 2.2. Клас `Queue` на `deque`

In [ ]:
class Queue:
    """Черга (FIFO): першим прийшов — першим обслужений. Усередині deque."""

    def __init__(self, items=()):
        self._items = deque(items)

    def enqueue(self, item):
        self._items.append(item)          # у хвіст

    def dequeue(self):
        if not self._items:
            raise EmptyError("черга порожня")
        return self._items.popleft()      # з голови, O(1)

    def peek(self):
        if not self._items:
            raise EmptyError("черга порожня")
        return self._items[0]

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __iter__(self):
        """Від голови до хвоста — у порядку обслуговування."""
        return iter(self._items)

    def __repr__(self):
        return f"Queue({list(self._items)!r})"

In [ ]:
orders = Queue()
for number in [101, 102, 105]:
    orders.enqueue(number)
    print("enqueue", number, "→", orders)

print("першим везуть:", orders.dequeue(), "→", orders)
print("наступне:", orders.peek(), "| у черзі:", len(orders))

assert list(orders) == [102, 105] and len(orders) == 2
try:
    Queue().dequeue()
    raise AssertionError("порожня черга мала кинути EmptyError")
except EmptyError as error:
    print("з порожньої:", error)

101 прийшло першим — і першим поїхало. Інтерфейс `Queue` майже такий самий, як у `Stack`; різниця лише в тому, **з якого кінця** беремо.

### Приклад 2.3. Ті самі замовлення — стек і черга

Задача: побачити цю «одну деталь» на однакових даних. Чотири замовлення надходять о 09:00, 09:05, 09:10, 09:15 — кладемо їх і в стек, і в чергу, а потім обслуговуємо.

**🔮 Прогноз:** кого стек обслужить першим? Скільки чекатиме замовлення 09:00 у стеку?

<details>
<summary>Відповідь</summary>

Стек першим видасть 09:15 — останнє. Замовлення 09:00 — найстаріше — у стеку обслужать **останнім**: для клієнтів це несправедливо. Черга видасть 09:00, 09:05, 09:10, 09:15.

</details>

In [ ]:
INCOMING = ["09:00 #201", "09:05 #202", "09:10 #203", "09:15 #204"]


def serve_all(structure, put, take):
    """Покласти всі замовлення в структуру, потім забирати, доки не спорожніє."""
    for order in INCOMING:
        put(structure, order)
    served = []
    while structure:
        served.append(take(structure))
    return served

In [ ]:
by_stack = serve_all(Stack(), Stack.push, Stack.pop)
by_queue = serve_all(Queue(), Queue.enqueue, Queue.dequeue)
print("стек: ", by_stack)
print("черга:", by_queue)
assert by_queue == INCOMING and by_stack == INCOMING[::-1]

Стек перевернув порядок, черга його зберегла. `serve_all` не знає, з якою структурою працює: ми передали їй **методи як функції** (`Stack.push`, `Queue.dequeue` — урок 18), а `while structure:` спрацював завдяки `__bool__` в обох класах.

### Приклад 2.4. `deque(maxlen=…)`: останні N адрес

Задача: оператор хоче бачити **3 останні** введені адреси. Невелика зміна — `deque(maxlen=3)`: коли місця немає, новий елемент витісняє найстаріший з протилежного кінця, без жодного `if`.

In [ ]:
recent = deque(maxlen=3)
for address in ["Хрещатик, 1", "Хорива, 5", "Оболонська, 12", "Хрещатик, 22", "Хорива, 5"]:
    recent.append(address)
    print(f"ввели {address!r:18} → {list(recent)}")

Після третьої адреси кожна нова виштовхує найдавнішу. «Хорива, 5» наприкінці потрапила в `deque` удруге — `deque` не прибирає дублікатів (це не множина). Такий самий механізм — у журналах «останні N подій».

### Приклад 2.5. Кур'єри по колу: `rotate`

Задача: роздати замовлення з черги трьом кур'єрам **по колу** — Оксана, Тарас, Ігор, знову Оксана… `deque.rotate(-1)` переставляє першого в кінець за `O(1)` — список кур'єрів сам стає «колом».

In [ ]:
def round_robin(orders, couriers):
    """Роздати замовлення з черги кур'єрам по колу; повертає [(кур'єр, замовлення)]."""
    shift = deque(couriers)
    plan = []
    while orders:
        courier = shift[0]
        plan.append((courier, orders.dequeue()))
        shift.rotate(-1)                  # перший — у кінець
        print(f"{courier:<7} ← {plan[-1][1]}   далі по колу: {list(shift)}")
    return plan

In [ ]:
plan = round_robin(Queue([101, 102, 103, 104, 105]), ["Оксана", "Тарас", "Ігор"])
assert [who for who, _ in plan] == ["Оксана", "Тарас", "Ігор", "Оксана", "Тарас"]

Кожне замовлення йде тому, хто зараз першим у колі, — і той переходить у кінець. Тут дві черги одночасно: `Queue` замовлень (FIFO) і `deque` кур'єрів, яку обертаємо. **Правило:** щойно треба брати з початку — бери `deque`, а не `list`.

---
## 3. Купа: найтерміновіше замовлення

Термінові замовлення везуть за **дедлайном**: першим — те, що має бути доставлене найраніше. Черга тут не підходить: замовлення з дедлайном 09:25 могло прийти пізніше за замовлення з дедлайном 09:40. Сортувати список після кожного нового замовлення — `O(n log n)` щоразу. **Купа** відповідає на одне питання — «який мінімальний?» — і робить це дешево: додати й зняти — `O(log n)`, подивитися — `O(1)`.

Купа — бінарне дерево, записане у звичайний список **по рівнях**: діти вузла `i` — `2*i + 1` і `2*i + 2`, батько — `(i - 1) // 2`. Єдине правило: **батько не більший за дітей**, тож мінімум — у `items[0]`.

```text
                [0] 09:10
              /           \
        [1] 09:25        [2] 09:40
        /      \
  [3] 09:50  [4] 10:05

список: ["09:10", "09:25", "09:40", "09:50", "10:05"]
```

Час у форматі `"ГГ:ХХ"` можна порівнювати як рядки: `"09:25" < "09:40"` — так само, як час.

### Приклад 3.1. Дерево в списку

Задача: побачити, що «дерево» — це лише правило індексів. Функція друкує список по рівнях (рівень `k` — це `2**k` елементів) і для кожного вузла — його дітей.

In [ ]:
def show_levels(items):
    """Надрукувати список-купу по рівнях дерева і дітей кожного вузла."""
    level, start = 0, 0
    while start < len(items):
        end = start + 2 ** level
        print(f"рівень {level}: {items[start:end]}")
        level, start = level + 1, end
    for i, value in enumerate(items):
        kids = [items[c] for c in (2 * i + 1, 2 * i + 2) if c < len(items)]
        if kids:
            print(f"  [{i}] {value} → діти {kids}")

In [ ]:
show_levels(["09:10", "09:25", "09:40", "09:50", "10:05"])

Жодних посилань між вузлами: діти `[0]` — це `[1]` і `[2]`, діти `[1]` — `[3]` і `[4]`. Кожен батько не більший за своїх дітей — отже, це купа. А між гілками порядку немає: `09:40` (рівень 1) купа ніколи не порівнює з `09:50` і `10:05` — вони діти **іншого** вузла. Саме тому купа дешевша за повне сортування.

**🔮 Прогноз:** список-купа `["09:40", "10:05"]`. Додаємо `"09:25"`. Де він опиниться і яким стане список?

<details>
<summary>Відповідь</summary>

Спершу в кінці (індекс 2). Його батько — індекс `(2 - 1) // 2 = 0`, `"09:40"`. `"09:25" < "09:40"` — обмін. Індекс 0 — вершина, стоп: `["09:25", "10:05", "09:40"]`.

</details>

**Картка класу `MinHeap`**

| Запитання | Відповідь | У коді |
|---|---|---|
| Що це за річ? | купа термінових замовлень з найменшим зверху | `class MinHeap` |
| Що знає? | елементи у списку-дереві і функцію порівняння | `self._items`, `self._key` |
| Що вміє? | додати, зняти мінімум, подивитися мінімум | `push`, `pop`, `peek` |
| Які правила тримає? | батько не більший за дітей — після **кожної** операції | `_sift_up`, `_sift_down` |
| Чого не робить? | не тримає повного порядку, не шукає довільний елемент | немає `sort`, `__contains__` |

- **push**: поставити новий елемент у кінець списку й **піднімати** (sift up), доки батько більший за нього;
- **pop**: забрати вершину, на її місце поставити останній елемент і **опускати** (sift down) до меншої дитини, доки та менша.

Кожен крок — один рівень дерева, а рівнів `log₂ n`. `key` — функція, за якою порівнюємо елементи (як `key` у `sorted`); порівняння — в одному методі `_less`.

### Приклад 3.2. Клас `MinHeap`

In [ ]:
class MinHeap:
    """Купа з мінімумом на вершині: push і pop — O(log n), peek — O(1)."""

    def __init__(self, items=(), key=None):
        self._key = key if key is not None else (lambda item: item)
        self._items = []
        for item in items:
            self.push(item)

    def push(self, item):
        self._items.append(item)
        self._sift_up(len(self._items) - 1)

    def pop(self):
        if not self._items:
            raise EmptyError("купа порожня")
        top = self._items[0]
        last = self._items.pop()
        if self._items:
            self._items[0] = last
            self._sift_down(0)
        return top

    def peek(self):
        if not self._items:
            raise EmptyError("купа порожня")
        return self._items[0]

    def _less(self, i, j):
        return self._key(self._items[i]) < self._key(self._items[j])

    def _sift_up(self, i):
        """Новий елемент піднімається, доки батько більший за нього."""
        while i > 0:
            parent = (i - 1) // 2
            if not self._less(i, parent):
                break
            self._items[i], self._items[parent] = self._items[parent], self._items[i]
            i = parent

    def _sift_down(self, i):
        """Елемент на вершині опускається до меншої дитини, доки вона менша за нього."""
        n = len(self._items)
        while True:
            smallest = i
            for child in (2 * i + 1, 2 * i + 2):
                if child < n and self._less(child, smallest):
                    smallest = child
            if smallest == i:
                break
            self._items[i], self._items[smallest] = self._items[smallest], self._items[i]
            i = smallest

    def __len__(self):
        return len(self._items)

    def __bool__(self):
        return bool(self._items)

    def __repr__(self):
        return f"MinHeap({self._items!r})"

In [ ]:
urgent = MinHeap()
for deadline in ["09:40", "10:05", "09:25"]:
    urgent.push(deadline)
    print("push", deadline, "→", urgent)

assert urgent._items == ["09:25", "10:05", "09:40"]
print("найтерміновіше:", urgent.pop(), "→", urgent)

Третій `push` — рівно те, що в прогнозі: `09:25` піднявся на вершину, а `09:40` опустився на його місце. Після `pop` на вершині вже `09:40`: останній елемент (`09:40`) став на вершину, порівнявся з дитиною `10:05` і лишився там.

### Приклад 3.3. Підйом по кроках

Задача: побачити кожне порівняння всередині `_sift_up`. Це та сама логіка, що й у методі, але окремою функцією над звичайним списком і з друком кожного кроку.

In [ ]:
def sift_up_traced(items, i):
    """_sift_up з друком: елемент items[i] піднімається, доки батько більший."""
    while i > 0:
        parent = (i - 1) // 2
        print(f"  [{i}] {items[i]} проти батька [{parent}] {items[parent]}", end=" → ")
        if not items[i] < items[parent]:
            print("батько менший, стоп")
            return
        items[i], items[parent] = items[parent], items[i]
        print("обмін:", items)
        i = parent
    print("  дійшли до вершини")

In [ ]:
heap = ["09:10", "09:25", "09:40", "09:50", "10:05"]
for deadline in ["09:30", "09:05"]:
    heap.append(deadline)
    print(f"push {deadline}: у кінець, індекс {len(heap) - 1} → {heap}")
    sift_up_traced(heap, len(heap) - 1)

`09:30` на індексі 5 порівнявся з батьком `[2] 09:40`, обмінявся, потім з `[0] 09:10` — і зупинився: два порівняння. `09:05` пройшов шлях 6 → 2 → 0 до самої вершини. Навіть у найгіршому разі кроків стільки, скільки рівнів дерева, а не скільки елементів.

### Приклад 3.4. Знімати мінімум, доки купа не спорожніє

**🔮 Прогноз:** у купу поклали `09:40, 10:05, 09:25, 09:50, 09:10`. У якому порядку їх видасть `pop()`? Чи буде внутрішній список відсортованим після кожного кроку?

<details>
<summary>Відповідь</summary>

`pop()` видасть 09:10, 09:25, 09:40, 09:50, 10:05 — за зростанням. А внутрішній список відсортованим бути **не зобов'язаний**: гарантовано лише, що мінімум — на `[0]`.

</details>

In [ ]:
urgent = MinHeap(["09:40", "10:05", "09:25", "09:50", "09:10"])
print("купа:       ", urgent)
while urgent:
    print(f"pop → {urgent.pop()} | лишилось {urgent}")

Кожен `pop` віддає мінімум з тих, що залишились, — отже, послідовність `pop` виходить відсортованою. Це **сортування купою** (heapsort). Подивись на внутрішні списки: наприклад, `['09:25', '09:50', '09:40', '10:05']` — не відсортований, але правило «батько не більший за дітей» виконано.

### Приклад 3.5. Суддя — `sorted` і `heapq`

Задача: переконатися, що `MinHeap` правильний не на трьох прикладах, а на сотнях. Прийом з уроку 26: повільне й очевидне (`sorted`) — суддя для швидкого. Ще один суддя — стандартний `heapq`. `random.Random(29)` дає ті самі «випадкові» набори при кожному запуску.

In [ ]:
rng = random.Random(29)
for trial in range(300):
    items = [rng.randint(0, 50) for _ in range(rng.randint(0, 40))]
    ours = MinHeap(items)
    std = list(items)
    heapq.heapify(std)
    assert [ours.pop() for _ in range(len(ours))] == sorted(items) == [heapq.heappop(std) for _ in range(len(std))]
print("300 випадкових наборів: MinHeap збігається з sorted і heapq")

Набори бувають порожні, з дублікатами, з одного елемента — усі граничні випадки покриваються самі. Такий самий тест є в проєкті `dispatch`.

### Приклад 3.6. `heapq`: купа зі стандартної бібліотеки

У Python купа — модуль `heapq`: функції, що працюють зі **звичайним списком**. Щоб купа впорядковувала замовлення за дедлайном, кладуть кортежі `(дедлайн, замовлення)`: кортежі порівнюються поелементно, тож вирішує перший елемент.

In [ ]:
URGENT_ORDERS = [("09:40", "#103"), ("09:25", "#104"), ("10:05", "#105"), ("09:10", "#106")]

In [ ]:
queue = []
for deadline, order in URGENT_ORDERS:
    heapq.heappush(queue, (deadline, order))
    print("heappush", (deadline, order), "→ вершина:", queue[0])

print("heappop →", heapq.heappop(queue), "| лишилось:", queue)

batch = list(URGENT_ORDERS)
heapq.heapify(batch)                     # перетворити готовий список на купу за O(n)
print("heapify:", batch)

`heapq.heappush` / `heappop` — ті самі `push`/`pop`, що в `MinHeap`, лише список тримаєш ти. `heapify` перетворює вже готовий список на купу на місці — швидше, ніж `n` разів `heappush`. Наш `MinHeap` робить те саме, але з `key=` замість кортежів і з інтерфейсом класу.

### Приклад 3.7. Рівні дедлайни в `heapq`: лічильник-«розв'язувач»

Задача: замовлення — словники, і в двох однаковий дедлайн 09:30. Коли перші елементи кортежів рівні, Python порівнює **другі** — а словники не порівнюються.

**🔮 Прогноз:** що станеться на другому `heappush`?

<details>
<summary>Відповідь</summary>

`TypeError: '<' not supported between instances of 'dict' and 'dict'`: дедлайни рівні, і `heapq` пробує порівняти самі словники. Виправлення — вставити між ними **порядковий номер** надходження: `(дедлайн, номер, замовлення)`. Номери різні, тож до словників справа не дійде.

</details>

In [ ]:
ORDER_A = {"id": 7, "address": "Хорива, 5"}
ORDER_B = {"id": 3, "address": "Поділ"}

In [ ]:
broken = []
heapq.heappush(broken, ("09:30", ORDER_A))
try:
    heapq.heappush(broken, ("09:30", ORDER_B))
except TypeError as error:
    print("без лічильника:", error)

arrival = itertools.count()              # 0, 1, 2, ... — номер надходження
fixed = []
for order in (ORDER_A, ORDER_B):
    heapq.heappush(fixed, ("09:30", next(arrival), order))
print("з лічильником: ", [heapq.heappop(fixed) for _ in range(2)])

Лічильник робить кожен кортеж унікальним і заодно дає правило «при рівному дедлайні — хто раніше прийшов». Порядок серед рівних — це **рішення**, яке треба ухвалити явно; до нього повернемось у Задачі 3.

### Приклад 3.8. `key=` замість кортежів і `nsmallest`

Задача: ті самі словники, але без кортежів — `MinHeap(key=…)` порівнює за функцією, як `sorted(key=…)`. А для разового «покажи 2 найтерміновіші» не треба навіть купи-об'єкта: `heapq.nsmallest`.

In [ ]:
ORDERS = [
    {"id": 103, "deadline": "09:40"},
    {"id": 104, "deadline": "09:25"},
    {"id": 105, "deadline": "10:05"},
    {"id": 106, "deadline": "09:10"},
]


def by_deadline(order):
    return order["deadline"]

In [ ]:
urgent = MinHeap(ORDERS, key=by_deadline)
print("peek:", urgent.peek())
print("усі по черзі:", [urgent.pop()["id"] for _ in range(len(urgent))])
print("2 найтерміновіші:", heapq.nsmallest(2, ORDERS, key=by_deadline))

Купа не знає, що таке «замовлення» чи «дедлайн», — їй досить функції `key`. Саме це робить `MinHeap` придатним для будь-яких даних (див. архітектуру пакета в розділі 6). `nsmallest(k, …)` — коли відповідь потрібна один раз; купа-об'єкт — коли замовлення надходять і забираються постійно.

---
## 4. Префіксне дерево: автодоповнення

Оператор набирає «Хр» — застосунок має підказати «Хрещатик, 1» і «Хрещатик, 22». Список адрес і `startswith` перевіряє **кожну** адресу. **Префіксне дерево** (trie) зберігає слова **по літерах**: слова зі спільним початком ділять одну гілку. Щоб знайти всі слова на «Хр», досить пройти дві літери від кореня.

```text
корінь
├── Х
│   ├── о → р → и → в → а ✓        «Хорива»
│   └── р ← пошук «Хр» зупиняється тут; усе нижче — відповідь
│       └── е → щ → а → т → и → к ✓  «Хрещатик»
└── О
    ├── б → о → л → о → н → ь ✓    «Оболонь»
    └── с → о → к → о → р → к → и ✓ «Осокорки»
```

Позначка ✓ (`is_word`) означає «тут закінчується ціле слово»: «Хр» — лише префікс, а не адреса.

### Приклад 4.1. Дерево з вкладених словників

Задача: побачити ідею без класів. Кожен вузол — словник `літера → вузол`; `setdefault` бере існуючий вузол або створює новий. Кінець слова позначимо ключем `"✓"`.

In [ ]:
root = {}
for word in ["кит", "кіт", "ко"]:
    node = root
    for letter in word:
        node = node.setdefault(letter, {})
    node["✓"] = True

In [ ]:
print(root)

Усі три слова ділять вузол `к`; далі гілки розходяться на `и`, `і`, `о`. Літера `"✓"` як ключ — хак: її не відрізнити від справжньої літери. Тому в класі вузол — окремий об'єкт із полем `is_word`.

**Картка класу `Trie`**

| Запитання | Відповідь | У коді |
|---|---|---|
| Що це за річ? | словник адрес, упорядкований за літерами | `class Trie`, вузли `_Node` |
| Що знає? | корінь дерева і кількість різних слів | `self._root`, `self._size` |
| Що вміє? | додати слово, перевірити ціле слово, знайти всі слова з префіксом | `add`, `__contains__`, `starts_with` |
| Які правила тримає? | дубль не збільшує кількість; префікс ≠ слово | `if not node.is_word`, перевірка `is_word` |
| Чого не робить? | не шукає з середини слова, не виправляє одруківки | лише префікси |

### Приклад 4.2. Класи `_Node` і `Trie`

`__slots__` у вузлі (урок 24) економить пам'ять: вузлів стільки, скільки різних префіксів, — тисячі. `_collect` — обхід дерева в глибину (рекурсія, урок 23); літери — за алфавітом, тож слова виходять відсортованими. `limit` зупиняє обхід, коли підказок досить.

In [ ]:
class _Node:
    __slots__ = ("children", "is_word")

    def __init__(self):
        self.children = {}      # літера -> _Node
        self.is_word = False


class Trie:
    """Префіксне дерево: пошук слова і префікса — O(довжина), незалежно від кількості слів."""

    def __init__(self, words=()):
        self._root = _Node()
        self._size = 0
        for word in words:
            self.add(word)

    def add(self, word):
        node = self._root
        for letter in word:
            node = node.children.setdefault(letter, _Node())
        if not node.is_word:
            node.is_word = True
            self._size += 1

    def _find(self, prefix):
        """Вузол, до якого веде префікс, або None."""
        node = self._root
        for letter in prefix:
            node = node.children.get(letter)
            if node is None:
                return None
        return node

    def __contains__(self, word):
        node = self._find(word)
        return node is not None and node.is_word

    def starts_with(self, prefix, limit=None):
        """Усі слова з цим префіксом в алфавітному порядку (не більше limit)."""
        node = self._find(prefix)
        if node is None:
            return []
        words = []
        self._collect(node, prefix, words, limit)
        return words

    def _collect(self, node, path, words, limit):
        if limit is not None and len(words) >= limit:
            return
        if node.is_word:
            words.append(path)
        for letter in sorted(node.children):
            self._collect(node.children[letter], path + letter, words, limit)

    def __len__(self):
        return self._size

    def __iter__(self):
        return iter(self.starts_with(""))

    def __repr__(self):
        return f"Trie({len(self)} слів)"

In [ ]:
ADDRESSES = ["Хрещатик, 1", "Хрещатик, 22", "Хорива, 5", "Оболонська, 12", "Хрещатик, 1"]
addresses = Trie(ADDRESSES)
print(addresses, "— дубль «Хрещатик, 1» не порахувався")
print("Хр →", addresses.starts_with("Хр"))
print("Х  →", addresses.starts_with("Х"))
print("Ки →", addresses.starts_with("Ки"))
print("'Хорива, 5' in:", "Хорива, 5" in addresses, "| 'Хорива' in:", "Хорива" in addresses)
print("усі за алфавітом:", list(addresses))

assert addresses.starts_with("Хр") == ["Хрещатик, 1", "Хрещатик, 22"]
assert addresses.starts_with("Х") == ["Хорива, 5", "Хрещатик, 1", "Хрещатик, 22"]
assert addresses.starts_with("Ки") == []
assert "Хорива, 5" in addresses and "Хорива" not in addresses
assert len(addresses) == 4

- «Х» дав і «Хорива», і «Хрещатик»: `о` за алфавітом раніше за `р`.
- «Ки» — такої гілки немає, `_find` повернув `None`, а `starts_with` — порожній список.
- «Хорива» — шлях у дереві є, але `is_word` там `False`: адресу «Хорива» ніхто не додавав.

### Приклад 4.3. Надрукувати дерево

Задача: побачити структуру, яку будує `Trie`. Ланцюжки вузлів з однією дитиною друкуємо одним шматком («орива»), щоб дерево вміщалося на екрані; `✓` — кінець слова.

In [ ]:
def show_trie(node, indent=""):
    """Друкує дерево; ланцюжок вузлів з однією дитиною — одним рядком."""
    for letter in sorted(node.children):
        child, chunk = node.children[letter], letter
        while len(child.children) == 1 and not child.is_word:
            (next_letter, next_child), = child.children.items()
            chunk += next_letter
            child = next_child
        print(f"{indent}{chunk}{' ✓' if child.is_word else ''}")
        show_trie(child, indent + "    ")

In [ ]:
show_trie(Trie(["Хорива", "Хрещатик", "Хрещатик, 22", "Оболонь", "Осокорки"])._root)

«Хорива» і «Хрещатик» ділять одну літеру `Х`, «Оболонь» і «Осокорки» — `О`. «Хрещатик, 22» не має власної гілки від кореня: він продовжує вузол «Хрещатик ✓» чотирма символами `, 22`. Ось чому пошук за префіксом не залежить від кількості адрес: ми спускаємося лише по літерах префікса.

### Приклад 4.4. Скільки нових вузлів створює `add`

Задача: перевірити, що спільні префікси справді **діляться**. Невелика зміна — підклас `TracedTrie`, який перевизначає лише `add` і друкує, скільки вузлів пройшов існуючих, а скільки створив. Решта поведінки успадковується.

In [ ]:
class TracedTrie(Trie):
    def add(self, word):
        node, reused, created = self._root, 0, 0
        for letter in word:
            if letter in node.children:
                reused += 1
            else:
                node.children[letter] = _Node()
                created += 1
            node = node.children[letter]
        duplicate = node.is_word
        if not duplicate:
            node.is_word = True
            self._size += 1
        print(f"{word!r:16} пройшли {reused:2}, створили {created:2}" + ("  (дубль)" if duplicate else ""))

In [ ]:
traced = TracedTrie(["Хрещатик, 1", "Хрещатик, 22", "Хорива, 5", "Хрещатик, 1"])
print(traced)

«Хрещатик, 1» — 11 нових вузлів (дерево порожнє). «Хрещатик, 22» пройшов 10 існуючих — спільний початок «Хрещатик, » — і створив лише 2 вузли для `22`. «Хорива, 5» ділить лише `Х`: 1 пройдений, 8 нових. Дубль нічого не створив і не збільшив `len`. Чим більше адрес на одній вулиці, тим дешевше кожна наступна.

### Приклад 4.5. Підказки на кожне натискання

Задача: оператор набирає «Хрещатик, 1» літера за літерою, після кожної — не більше 3 підказок. Це звичайний `starts_with(prefix, limit=3)` у циклі.

In [ ]:
STREET_ADDRESSES = Trie(
    [f"Хрещатик, {n}" for n in (1, 2, 10, 12, 15, 22)]
    + ["Хорива, 5", "Хмельницького, 3", "Оболонська, 12"]
)

In [ ]:
typed = ""
for letter in "Хре":
    typed += letter
    print(f"{typed!r:8} → {STREET_ADDRESSES.starts_with(typed, limit=3)}")
for typed in ("Хрещатик, 1", "Хрещатик, 3"):
    print(f"{typed!r:8} → {STREET_ADDRESSES.starts_with(typed, limit=3)}")

Після «Х» — три перші за алфавітом з усіх вулиць на «Х», після «Хр» лишився тільки Хрещатик. «Хрещатик, 1» підказує 1, 10, 12 — і сам «Хрещатик, 1» перший: він і є ціле слово, і префікс інших. «Хрещатик, 3» — порожньо. Зверни увагу на порядок: `"10" < "2"` у рядках, тож «Хрещатик, 10» іде раніше за «Хрещатик, 2».

### Приклад 4.6. Дерево проти списку на 20 000 адрес

Задача: перевірити, що дерево справді швидше на великому довіднику. Однакове питання — «5 перших адрес з префіксом» — двома способами, по 200 разів для кожного префікса.

In [ ]:
STREETS = ["Хрещатик", "Хорива", "Оболонська", "Олегівська", "Антоновича", "Велика Васильківська",
           "Січових Стрільців", "Лук'янівська", "Межигірська", "Саксаганського"]
BIG_LIST = [f"{street}, {n}" for street in STREETS for n in range(1, 2001)]


def suggest_by_list(prefix, limit=5):
    return sorted(a for a in BIG_LIST if a.startswith(prefix))[:limit]


def measure(search, prefixes, repeat=200):
    start = time.perf_counter()
    for _ in range(repeat):
        for prefix in prefixes:
            search(prefix)
    return time.perf_counter() - start

In [ ]:
start = time.perf_counter()
BIG_TRIE = Trie(BIG_LIST)
print(f"{len(BIG_TRIE)} адрес; дерево побудовано за {time.perf_counter() - start:.2f} с")

prefixes = ["Хрещатик, 19", "Ол", "Саксаганського, 7"]
for prefix in prefixes:
    assert BIG_TRIE.starts_with(prefix, 5) == suggest_by_list(prefix)
    print(f"{prefix!r:20} → {BIG_TRIE.starts_with(prefix, 5)}")

by_list = measure(suggest_by_list, prefixes)
by_trie = measure(lambda p: BIG_TRIE.starts_with(p, 5), prefixes)
print(f"список: {by_list:.2f} с | дерево: {by_trie:.3f} с | дерево швидше в {by_list / by_trie:.0f} разів")

Відповіді однакові (`assert` це перевірив), а дерево — у десятки, а то й сотні разів швидше: список щоразу перебирає всі 20 000 адрес, дерево спускається на кілька літер і збирає 5 слів. Ціна — **побудова** дерева й пам'ять на вузли. **Правило:** дерево окупається, коли довідник будують раз, а питають багато разів (кожне натискання клавіші).

---
## 5. LRU-кеш: ціни маршрутів

Ціну поїздки між районами дає повільний картографічний сервіс. Ті самі маршрути питають знову й знову — результати варто **зберігати**. Але в кеші поміститься лише кілька записів. Кого викидати, коли місця немає?

**LRU** (least recently used) — викинути той запис, до якого **найдовше не зверталися**. `OrderedDict` пам'ятає порядок ключів: на початку — найдавніше використаний, у кінці — найсвіжіший. `move_to_end(key)` — «щойно використали», `popitem(last=False)` — викинути найдавніший; обидва `O(1)`.

```text
put Поділ→Оболонь   put Центр→Поділ          get Поділ→Оболонь         put Центр→Оболонь (місця 2)
[П→О]               [П→О, Ц→П]                [Ц→П, П→О]                [П→О, Ц→О]
                     ↑ найдавніший             ↑ найдавніший — Ц→П        Ц→П викинуто
```

### Приклад 5.1. Дві операції `OrderedDict`

In [ ]:
routes = OrderedDict()
routes["Поділ→Оболонь"] = 165
routes["Центр→Поділ"] = 105
routes["Центр→Оболонь"] = 195
print("додали три:            ", list(routes))

routes.move_to_end("Поділ→Оболонь")
print("Поділ→Оболонь в кінець:", list(routes))

oldest = routes.popitem(last=False)
print("викинули найдавніший:  ", oldest, "→", list(routes))

`move_to_end` переставив ключ у кінець, не змінюючи значення; `popitem(last=False)` зняв **перший** ключ разом зі значенням. З цих двох операцій і складається весь LRU.

**🔮 Прогноз:** кеш на 2 записи: `put(A)`, `put(B)`, `get(A)`, `put(C)`. Хто залишиться?

<details>
<summary>Відповідь</summary>

`A` і `C`. `get(A)` зробив `A` найсвіжішим, тож найдавніше використаним став `B` — його й викинули. Без `get(A)` викинули б `A`.

</details>

### Приклад 5.2. FIFO проти LRU на тих самих запитах

Задача: побачити, навіщо `move_to_end`. Одна функція моделює кеш на 2 записи; параметр `lru` вирішує, чи переставляти ключ при влученні. Без перестановки кеш викидає **найдавніше доданий** — це FIFO-кеш. Запити: популярний маршрут «Поділ» питають через раз.

In [ ]:
REQUESTS = ["Поділ", "Центр", "Поділ", "Оболонь", "Поділ", "Лівобережна", "Поділ"]


def simulate(requests, capacity, lru):
    """Прогнати запити через кеш на OrderedDict; друкує кожен крок, повертає кількість влучань."""
    cache, hits = OrderedDict(), 0
    for key in requests:
        if key in cache:
            hits += 1
            status = "влучання"
            if lru:
                cache.move_to_end(key)        # ← єдина різниця між LRU і FIFO
        else:
            status = "промах"
            cache[key] = True
            if len(cache) > capacity:
                status += f", викинуто {cache.popitem(last=False)[0]}"
        print(f"  {key:<12} {status:<30} {list(cache)}")
    return hits

In [ ]:
print("LRU:")
lru_hits = simulate(REQUESTS, capacity=2, lru=True)
print("FIFO:")
fifo_hits = simulate(REQUESTS, capacity=2, lru=False)
print(f"влучань: LRU {lru_hits}, FIFO {fifo_hits}")

Дивись на четвертий запит («Оболонь»). У LRU «Поділ» щойно використали, тож викинуто «Центр». У FIFO «Поділ» доданий першим — і вилітає, хоча його питають найчастіше; на п'ятому запиті — зайвий промах. Тепер зрозуміло, чим поганий «спрощений» `get` без `move_to_end`: кеш перетворюється на FIFO. **Правило:** звернення до запису має **оновлювати** його вік.

### Приклад 5.3. Готовий кеш для функцій: `functools.lru_cache`

Для **функцій** кеш уже є — декоратор `lru_cache` з уроку 10. `maxsize=2` — той самий LRU на 2 записи; `cache_info()` показує влучання й промахи.

In [ ]:
DISTANCES_KM = {frozenset({"Поділ", "Оболонь"}): 7, frozenset({"Поділ", "Центр"}): 3}


@lru_cache(maxsize=2)
def estimate_price(origin, destination):
    print(f"    рахуємо {origin} → {destination}")
    return 60 + 15 * DISTANCES_KM[frozenset({origin, destination})]

In [ ]:
for route in [("Поділ", "Оболонь"), ("Центр", "Поділ"), ("Поділ", "Оболонь")]:
    print(" → ".join(route), estimate_price(*route), "грн")
print(estimate_price.cache_info())

«рахуємо …» надруковано лише двічі: третій виклик узято з кешу, тіло функції не виконувалось. `CacheInfo(hits=1, misses=2, …)` — те саме, що видно в рядках.

### Приклад 5.4. Повільний сервіс: виграш кешу і `cache_clear`

Задача: поміряти виграш. `time.sleep(0.2)` імітує мережевий запит до картографічного сервісу.

In [ ]:
@lru_cache(maxsize=32)
def slow_price(origin, destination):
    time.sleep(0.2)                       # «запит у мережу»
    return 60 + 15 * DISTANCES_KM[frozenset({origin, destination})]


def timed_price(origin, destination):
    start = time.perf_counter()
    price = slow_price(origin, destination)
    print(f"{origin} → {destination}: {price} грн за {time.perf_counter() - start:.3f} с")

In [ ]:
timed_price("Поділ", "Оболонь")
timed_price("Поділ", "Оболонь")
timed_price("Оболонь", "Поділ")
slow_price.cache_clear()
print("кеш очищено:", slow_price.cache_info())
timed_price("Поділ", "Оболонь")

Другий виклик — майже миттєвий. Третій («Оболонь → Поділ») знову повільний: для `lru_cache` це **інші аргументи**, хоча відстань та сама. Після `cache_clear()` усе рахується заново.

Власний `LRUCache` потрібен, коли кешуємо не функцію, а **дані** всередині об'єкта: ключі обираємо самі (наприклад, можна нормалізувати маршрут), а ємність і статистику бачить сервіс.

**Картка класу `LRUCache`**

| Запитання | Відповідь | У коді |
|---|---|---|
| Що це за річ? | пам'ять на кілька останніх цін маршрутів | `class LRUCache` |
| Що знає? | ємність, записи від найдавнішого до найсвіжішого, статистику | `capacity`, `self._data` (`OrderedDict`), `hits`, `misses` |
| Що вміє? | дати значення, записати значення | `get`, `put` |
| Які правила тримає? | не більше `capacity` записів; звернення робить запис найсвіжішим | `popitem(last=False)`, `move_to_end` |
| Чого не робить? | не рахує ціну сама, не знає про маршрути | ціну дає сервіс (розділ 6) |

### Задача 2. `get` і `put` класу `LRUCache`

Решту класу написано. Допиши два методи:

- `get(key, default=None)`: ключа немає — `misses += 1`, повернути `default`; є — `hits += 1`, `move_to_end(key)`, повернути значення;
- `put(key, value)`: записати; існуючий ключ — теж у кінець; якщо записів стало більше за `capacity` — викинути найдавніший.

Перевірка — у наступній клітинці.

In [ ]:
class LRUCache:
    """Кеш на capacity записів; викидає той, до якого найдовше не зверталися."""

    def __init__(self, capacity):
        if capacity < 1:
            raise ValueError("capacity має бути хоча б 1")
        self.capacity = capacity
        self._data = OrderedDict()           # початок — найдавніший, кінець — найсвіжіший
        self.hits = 0
        self.misses = 0

    # YOUR CODE HERE
    pass

    def __contains__(self, key):
        return key in self._data

    def __len__(self):
        return len(self._data)

    def keys(self):
        return list(self._data)

    def __repr__(self):
        return f"LRUCache(capacity={self.capacity}, keys={self.keys()!r})"

In [ ]:
cache = LRUCache(2)
cache.put("A", 1)
cache.put("B", 2)
print("put A, put B:", cache)
assert cache.get("A") == 1
print("get A:       ", cache)
cache.put("C", 3)
print("put C:       ", cache)
assert cache.keys() == ["A", "C"], "get(A) мав зробити A найсвіжішим — викинути треба було B"
assert cache.get("B", "немає") == "немає"
assert (cache.hits, cache.misses) == (1, 1), "hits/misses: один get(A) — влучання, один get(B) — промах"
cache.put("A", 10)
print("put A=10:    ", cache)
assert cache.keys() == ["C", "A"] and cache.get("A") == 10, "оновлення існуючого ключа теж переносить його в кінець"
print("✅ Задача 2 розв'язана")

---
## 6. Пакет `dispatch` і його тести

Готовий пакет — у папці `dispatch_project` поруч з цим ноутбуком. У Colab ноутбук відкривається **без** файлів репозиторію, тому клітинка нижче в такому разі завантажує лише цю папку з GitHub (`git clone --sparse`, близько пів мегабайта).

Структури — інструменти. Сервіс `Dispatcher` **складається** з них (композиція, урок 21): кожне поле — окрема структура з однією відповідальністю.

```text
dispatch_project/
├── pytest.ini
├── README.md
├── dispatch/
│   ├── __init__.py          публічний інтерфейс: Order, Dispatcher
│   ├── __main__.py          python -m dispatch — демо
│   ├── models.py            Order
│   ├── pricing.py           estimate_price
│   ├── service.py           Dispatcher
│   └── structures/          Stack, Queue, MinHeap, Trie, LRUCache, EmptyError
└── tests/                   57 тестів

Dispatcher
├── _regular  : Queue     звичайні замовлення — FIFO
├── _urgent   : MinHeap   термінові — за дедлайном
├── _returned : Stack     скасовані — видаємо першими
├── _history  : Stack     для undo
├── _addresses: Trie      автодоповнення
└── _prices   : LRUCache  ціни маршрутів
```

In [ ]:
import os
import subprocess
import sys

PROJECT = "dispatch_project"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"
PATH_IN_REPO = "module_2/lessons/lesson_29_practicum_data_structures/dispatch_project"

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
    subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", PATH_IN_REPO], check=True)
    PROJECT = os.path.join("course_repo", PATH_IN_REPO)

print("Проєкт:", PROJECT)
print(sorted(os.listdir(os.path.join(PROJECT, "dispatch"))))

### Приклад 6.1. Пакет як програма і його тести

`run(...)` запускає Python у папці проєкту — як у терміналі `python -m dispatch`. `python -m dispatch` виконує `dispatch/__main__.py`: так пакет стає програмою. Шлях до проєкту додаємо в `sys.path`, щоб далі імпортувати `dispatch` прямо в ноутбук.

In [ ]:
def run(*args):
    """Запустити команду Python у папці проєкту, як у терміналі."""
    result = subprocess.run([sys.executable, *args], cwd=PROJECT, capture_output=True, text=True)
    print((result.stdout + result.stderr)[-2500:])
    return result.returncode


sys.path.insert(0, PROJECT)

In [ ]:
run("-m", "dispatch")

Останній рядок демо — статистика кешу: маршрут «Поділ → Оболонь» удруге взято з пам'яті (`hits: 1`).

Тести пакета — pytest (урок 26). Якщо pytest не встановлено, спершу: `pip install pytest` у терміналі (у Colab — клітинка `!pip install pytest`; зазвичай він уже є).

In [ ]:
code = run("-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no")
assert code == 0

57 тестів перевіряють кожну структуру окремо (зокрема купу проти `heapq` і `sorted`) і сервіс цілком. Код повернення `0` — усі пройшли.

### Приклад 6.2. `Dispatcher` у ноутбуці крок за кроком

Задача: побачити, яка структура відповідає на яке питання. Імпортуємо справжні `Order` і `Dispatcher` з пакета й після кожної дії дивимось, скільки замовлень чекає.

**🔮 Прогноз:** звичайні #101, #102 прийшли раніше за термінові #103 (09:40) і #104 (09:25). Кого отримає перший кур'єр?

<details>
<summary>Відповідь</summary>

#104: спершу видаються повернуті (`Stack`), потім термінові з купи за дедлайном — 09:25 раніше за 09:40, — і лише потім черга звичайних.

</details>

In [ ]:
from dispatch import Dispatcher, Order
from dispatch.pricing import estimate_price as package_price

CITY = ["Хрещатик, 1", "Хрещатик, 22", "Хорива, 5", "Оболонська, 12"]

In [ ]:
d = Dispatcher(CITY)
d.receive(Order(101, "Оболонська, 12"))
d.receive(Order(102, "Хорива, 5"))
d.receive(Order(103, "Хрещатик, 22", deadline="09:40"), urgent=True)
d.receive(Order(104, "Хрещатик, 1", deadline="09:25"), urgent=True)
print(d)

for courier in ["Оксана", "Тарас", "Ігор"]:
    print(f"{courier} ← {d.assign(courier)} | {d}")
print("скасовано:", *d.undo(), "|", d)
print(f"Марія ← {d.assign('Марія')} | у Ігоря: {d.assigned_to('Ігор')}")
print("Хр… →", d.suggest("Хр"))

- Термінові йдуть першими й **за дедлайном**, а не за часом надходження: #104 (09:25) раніше за #103 (09:40).
- Скасоване #101 не загубилося і не стало в кінець черги: стек `_returned` видав його Марії наступним.
- `assigned_to("Ігор")` порожній: `undo` прибрав замовлення і зі списку кур'єра.

### Приклад 6.3. Кеш цін і «залежність параметром»

Задача: переконатися, що повторний маршрут не рахується вдруге. `Dispatcher` отримує функцію ціни **параметром** (`price_function=`), тож замість справжньої можна підставити обгортку, яка записує кожен виклик, — так роблять і тести (урок 26).

In [ ]:
calls = []


def counting_price(origin, destination):
    calls.append((origin, destination))
    return package_price(origin, destination)

In [ ]:
d = Dispatcher(price_function=counting_price, cache_size=2)
for route in [("Поділ", "Оболонь"), ("Центр", "Поділ"), ("Поділ", "Оболонь"), ("Центр", "Оболонь"), ("Центр", "Поділ")]:
    print(" → ".join(route), d.route_price(*route), "грн")
print("справжніх розрахунків:", calls)
print(d.cache_stats)

П'ять запитів — чотири розрахунки. Третій запит («Поділ → Оболонь») узято з кешу. А п'ятий («Центр → Поділ») знову рахувався: кеш на 2 записи, і після «Центр → Оболонь» він уже викинув найдавніше використаний «Центр → Поділ». `Dispatcher` не змінювали — змінили лише те, що передали в конструктор.

### Приклад 6.4. Рівні дедлайни: хто перший?

Задача: два термінові замовлення #3 і #7 з дедлайном 09:30, #3 прийшло першим. Купа порівнює лише дедлайн, тож для неї ці замовлення **рівні** — і порядок між ними випадковий з погляду користувача.

In [ ]:
TIE = [Order(3, "Поділ", "09:30"), Order(9, "Хрещатик, 1", "09:10"), Order(7, "Хорива, 5", "09:30")]

In [ ]:
d = Dispatcher()
for order in TIE:
    d.receive(order, urgent=True)
print("видано:", [d.assign("Оксана").order_id for _ in range(len(TIE))])

Після #9 (09:10) купа видала **#7**, хоча #3 прийшло раніше й має менший номер. Причина — механіка `_sift_down`: рівні елементи не обмінюються, тож порядок між ними залежить від того, де вони опинились у списку. Правило «при рівному дедлайні — менший номер» треба задати **явно**.

### Задача 3. Рівні дедлайни: ключ-кортеж

Напиши функцію-ключ `urgent_key(order)` так, щоб купа видавала раніший дедлайн першим, а при рівному — менший номер замовлення.

Кортежі порівнюються поелементно: `("09:30", 3) < ("09:30", 7)` (той самий прийом, що й лічильник у прикладі 3.7). У пакеті це зміна одного рядка — `key=` при створенні `_urgent` у `service.py`; тут перевіряємо на справжньому `Order` і нашому `MinHeap`, в обох порядках надходження.

Перевірка — у наступній клітинці.

In [ ]:
def urgent_key(order):
    # YOUR CODE HERE
    pass


In [ ]:
for arrival in (TIE, TIE[::-1]):
    heap = MinHeap(arrival, key=urgent_key)
    result = [heap.pop().order_id for _ in range(len(arrival))]
    print("надійшли", [o.order_id for o in arrival], "→ видано", result)
    assert result == [9, 3, 7], "при рівному дедлайні першим має йти менший номер"
print("✅ Задача 3 розв'язана")

---
## Підсумок

| Структура | Принцип | Операції | Складність | У стандартній бібліотеці |
|---|---|---|---|---|
| Стек | LIFO | `push`, `pop`, `peek` | `O(1)` | `list`: `append`, `pop()` |
| Черга | FIFO | `enqueue`, `dequeue` | `O(1)` | `collections.deque`: `append`, `popleft` |
| Купа | мінімум на вершині | `push`, `pop` / `peek` | `O(log n)` / `O(1)` | `heapq` |
| Префіксне дерево | спільні префікси — спільні гілки | `add`, `in`, `starts_with` | `O(довжина)` | немає; близько — `dict` вкладених `dict` |
| LRU-кеш | викидати найдавніше використане | `get`, `put` | `O(1)` | `functools.lru_cache`, `OrderedDict` |

---
## Самоперевірка

1. Стек чи черга: скасування дій? обслуговування клієнтів у порядку дзвінків?
2. Чому `deque` для черги, а не `list`?
3. Що гарантує правило купи і чого не гарантує?
4. Навіщо в префіксному дереві `is_word`?
5. Який рядок робить кеш LRU, а не FIFO?
6. Чому `heapq.heappush` упав на кортежах `(дедлайн, словник)` з рівними дедлайнами і як це виправити?
7. Що робить `deque(maxlen=3)`, коли додаєш четвертий елемент?
8. Коли брати `functools.lru_cache`, а коли — власний `LRUCache`?
9. Чому `Dispatcher` містить структури як поля, а не наслідується від них?

<details>
<summary>Відповіді</summary>

1. Скасування — стек (останнє першим); дзвінки — черга (перше першим).
2. `list.pop(0)` зсуває всі елементи — `O(n)`; `deque.popleft()` — `O(1)`.
3. Мінімум на вершині; повного порядку між гілками — ні.
4. Щоб відрізнити ціле слово від префікса («Хорива» — префікс, «Хорива, 5» — адреса).
5. `self._data.move_to_end(key)` у `get`.
6. При рівних перших елементах Python порівнює другі, а словники не порівнюються. Між ними вставляють лічильник надходження: `(дедлайн, номер, замовлення)`.
7. Викидає найстаріший елемент з протилежного кінця — без жодного `if`.
8. `lru_cache` — для результатів функції за її аргументами. Власний — коли кешуємо дані всередині об'єкта: ключі обираємо самі, а ємність і статистику бачить сервіс.
9. Диспетчерська — не купа і не стек: вона **користується** ними. Композиція дає змогу замінити будь-яку структуру, не зачіпаючи решту, і тестувати структури окремо.

</details>

## Далі

- Книга: [Урок 29](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_29/) — покрокові схеми кожної структури й архітектура пакета.
- Проєкт `dispatch_project/`: README з розділом «Спробуй зламати» — змінюй код і дивись, який тест падає.
- **Модуль 3** — бази даних; в уроці 31 Redis дасть ті самі черги, стеки, черги з пріоритетом і LRU-кеш — поза програмою.